In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.sparse import coo_matrix, csr_matrix, eye as speye
from scipy.sparse.linalg import eigsh

In [2]:
# load and inspect
arr = np.load('data/node_features_topo.npy', allow_pickle=False)

print(f'shape: {arr.shape}')
print(f'dtype: {arr.dtype}')
print(f'nans:  {np.isnan(arr).sum()}')
print(f'infs:  {np.isinf(arr).sum()}')
print()

shape: (2041172, 6)
dtype: float32
nans:  0
infs:  0



In [3]:
col_labels = ['elevation','slope','aspect_sin','aspect_cos','curvature','log_acc']

print(f'{"Col":>4}  {"Feature":>12}  {"Min":>12}  {"Max":>12}  {"Mean":>12}  {"Std":>12}')
print('-' * 70)
for i, label in enumerate(col_labels):
    col = arr[:, i]
    print(f'{i:>4}  {label:>12}  {col.min():>12.4f}  {col.max():>12.4f}  {col.mean():>12.4f}  {col.std():>12.4f}')

 Col       Feature           Min           Max          Mean           Std
----------------------------------------------------------------------
   0     elevation      811.2109      987.2007      861.5251       38.7195
   1         slope        0.0178       82.3289       15.4682       10.0765
   2    aspect_sin       -1.0000        1.0000       -0.0190        0.7140
   3    aspect_cos       -1.0000        1.0000        0.1048        0.6920
   4     curvature       -0.1547        0.2178        0.0013        0.0543
   5       log_acc        0.6931       13.5554        2.2006        1.5978


In [10]:
# load and inspect index map
# note: gdalinfo reports width x height -- raster format
# array format is height (rows) x width (columns)

index_map = np.load('data/node_index_mosext.npy')

print(f'idx map shape: {index_map.shape}')
print(f'dtype:         {index_map.dtype}')
print()

nrows, ncols = index_map.shape
valid_mask   = index_map >= 0

print(f'full grid:     {nrows}x{ncols} = {nrows*ncols:,}')
print(f'valid pxls:    {valid_mask.sum():,}')
print(f'invalid (-1):  {(~valid_mask).sum():,}')
print(f'coverage:      {100 * valid_mask.sum() / (nrows*ncols):.1f}%')
print()

# confirm # valid pixels matches nodes in feature array
assert valid_mask.sum() == arr.shape[0], \
    f'mismatch: idx map has {valid_mask.sum()} val pixels but arr has {arr.shape[0]} nodes'
print('assertion passed: valid pixel count matches nodes in feature array')

idx map shape: (1875, 1957)
dtype:         int32

full grid:     1875x1957 = 3,669,375
valid pxls:    2,041,172
invalid (-1):  1,628,203
coverage:      55.6%

assertion passed: valid pixel count matches nodes in feature array


In [11]:
# gdalinfo reports (width, height) = (1957, 1875)
# numpy stores   (nrows, ncols)    = (1875, 1957)
# we follow numpy convention throughout

NROWS = 1875   # height, north-south extent
NCOLS = 1957   # width,  east-west extent
RES_M = 1.017  # pixel size in metres

print(f'Grid convention : {NROWS} rows (N-S) x {NCOLS} cols (E-W)')
print(f'Pixel resolution: {RES_M} m')
print(f'Approx extent   : {NROWS * RES_M / 1000:.2f} km (N-S) x {NCOLS * RES_M / 1000:.2f} km (E-W)')

Grid convention : 1875 rows (N-S) x 1957 cols (E-W)
Pixel resolution: 1.017 m
Approx extent   : 1.91 km (N-S) x 1.99 km (E-W)


In [13]:
# build 8-connectivity edge list
# note: (potential) edge-list only
# not weighted adjacency matrix

# 8-connectivity offsets
# all combinations of {-1, 0, 1}^2
# except (0,0)
offsets = [(-1,-1),(-1,0),(-1,1),
           ( 0,-1),        (0,1),
           ( 1,-1),( 1,0),( 1,1)]

rows_i, rows_j = [], []

for di, dj in offsets:
    
    # first set the offset
    # taking -di and -dj with min and max ensures in bounds
    r0, r1 = max(0, -di), min(NROWS, NROWS - di)
    c0, c1 = max(0, -dj), min(NCOLS, NCOLS - dj)

    # slice index map for source pixels and their neighbors
    src = index_map[r0:r1, c0:c1]
    nbr = index_map[r0+di:r1+di, c0+dj:c1+dj]

    # keep only pairs where source and neighbor are valid
    valid = (src >= 0) & (nbr >= 0)

    # store these in sequence
    rows_i.append(src[valid])
    rows_j.append(nbr[valid])

# convert to 1D vectors
edge_i = np.concatenate(rows_i)
edge_j = np.concatenate(rows_j)

# now for k in {N} number of nodes,
# edge_i[k] and edge_j[k] together
# describe the k-th edge
# e.g. if edge_i[100] = 42 and edge_j[100] = 127
# then the graph has a directed edge (42, 127)

print(f'Total directed edges : {len(edge_i):,}')
print(f'Total undirected edges: {len(edge_i) // 2:,}')
print()

# Sanity checks
print(f'Min node index in edges: {min(edge_i.min(), edge_j.min())}')
print(f'Max node index in edges: {max(edge_i.max(), edge_j.max())}')
print(f'Expected max           : {arr.shape[0] - 1}')

Total directed edges : 16,306,088
Total undirected edges: 8,153,044

Min node index in edges: 0
Max node index in edges: 2041171
Expected max           : 2041171


In [15]:
# load dDEM and align to array
import rasterio

with rasterio.open('data/CanDiff20002025_mosext.tif') as src:
    ddem_full = src.read(1) # shape (1875, 1957)
    ddem_full = -ddem_full # sign convention (-) means ice loss
    ddem_nodata = - src.nodata # modify along with sign flip
    print(f'ddem raster shape: {ddem_full.shape}')
    print(f'ddem nodata value: {ddem_nodata}')
    print(f'ddem dtype:        {ddem_full.dtype}')

print()

# replace nodata with NaN for safety
if ddem_nodata is not None:
    ddem_full = ddem_full.astype(np.float32)
    ddem_full[ddem_full == ddem_nodata] = np.nan

print(f'ddem range: [{np.nanmin(ddem_full):.2f}, {np.nanmax(ddem_full):.2f}] m')
print(f'ddem nans:  {np.isnan(ddem_full).sum():,}')

# extract valid pixels in node order using index map
# index_map[r,c] = k means node k corresponds to grid position (r,c)
# we need the inverse: for node k, what is (r,c)?

# row and col numbers corresponding to valid pixels
valid_r, valid_c = np.where(index_map >= 0)
# node numbers corresponding to valid rows and cols
node_order = index_map[valid_r, valid_c]

# sort valid node numbers to align with arr
sort_idx = np.argsort(node_order)
valid_r = valid_r[sort_idx]
valid_c = valid_c[sort_idx]
# these are now "shuffled" but aligned with arr

ddem_nodes = ddem_full[valid_r, valid_c]

print(f'ddem_nodes shape: {ddem_nodes.shape}')
print(f'ddem_nodes range: [{np.nanmin(ddem_nodes):.2f}, {np.nanmax(ddem_nodes):.2f}] m')
print(f'ddem_nodes nans:  {np.isnan(ddem_nodes).sum():,}')

ddem raster shape: (1875, 1957)
ddem nodata value: 32767.0
ddem dtype:        float32

ddem range: [-88.04, 19.72] m
ddem nans:  1,637,268
ddem_nodes shape: (2041172,)
ddem_nodes range: [-88.04, 19.72] m
ddem_nodes nans:  9,065


In [16]:
### compute edge weights for three laplacians ###
###                                           ###

In [17]:
# 1. extract signal values at edge endpoints
# elevations
elev_i = arr[edge_i, 0]
elev_j = arr[edge_j, 0]

# aspect sin and cos
asin_i = arr[edge_i, 2]
asin_j = arr[edge_j, 2]
acos_i = arr[edge_i, 3]
acos_j = arr[edge_j, 3]

# diffDEM
ddem_i = ddem_nodes[edge_i]
ddem_j = ddem_nodes[edge_j]

print("signals extracted")

signals extracted


In [18]:
## 2.1 -- elevation laplacian weights
# gaussian kernel on elevation difference
# sigma ~ 10m => beta = 1/(2*10^2) = 0.005
BETA_ELEV = 0.005
w_elev = np.exp(-BETA_ELEV * (elev_i - elev_j)**2).astype(np.float32)

print("elev laplacian weights calculated")

elev laplacian weights calculated


In [19]:
## 2.2 -- aspect laplacian weights
# aspect encoded as (sin,cos) 
# squared euclidean distance in unit circle
# become dissimilarity measure
# max possible distance^2 = 4 (opposite sides on unit circle)
# sigma ~ 0.5 units => beta = 1/(2*0.5^2) = 2.0
BETA_ASPECT = 2.0
aspect_dist2 = (asin_i - asin_j)**2 + (acos_i - acos_j)**2
w_aspect = np.exp(-BETA_ASPECT * aspect_dist2).astype(np.float32)

print("aspect laplacian weights calculated")

aspect laplacian weights calculated


In [20]:
## 2.3 -- dDEM laplacian weights
# gaussian kernel on dDEM difference
# sigma ~ 5m => beta = 1/(2*5^2) = 0.02
# edges where either endpoint is nan get weight 0
BETA_DDEM = 0.02
ddem_diff2 = (ddem_i - ddem_j)**2
w_ddem = np.exp(-BETA_DDEM * ddem_diff2).astype(np.float32)

# edges touching nan pixels get weight 0
ddem_nan_mask = np.isnan(ddem_i) | np.isnan(ddem_j)
w_ddem[ddem_nan_mask] = 0.0

print("ddem laplacian weights calculated")

ddem laplacian weights calculated


In [21]:
print(f'Edge weight statistics:')
print(f'{"":12}  {"Min":>8}  {"Max":>8}  {"Mean":>8}  {"~0 edges":>10}')
print('-' * 52)
for name, w in [('w_elev', w_elev), ('w_aspect', w_aspect), ('w_ddem', w_ddem)]:
    near_zero = (w < 1e-4).sum()
    print(f'{name:12}  {w.min():>8.4f}  {w.max():>8.4f}  {w.mean():>8.4f}  {near_zero:>10,}')

Edge weight statistics:
                   Min       Max      Mean    ~0 edges
----------------------------------------------------
w_elev          0.3035    1.0000    0.9995           0
w_aspect        0.0003    1.0000    0.7666           0
w_ddem          0.0000    1.0000    0.9927      77,864


In [22]:
## more carefully -- empirical beta calibration
# beta -- sensitivity threshold - inverse of kernel bandwidth
# "how much does the feature change between adjacent pixels?"
# sigma - "effective sigma" variance measured in data
# beta = 1 / (2 * sigma^2)

In [23]:
# 0.1 compute actual pairwise difference at edges
elev_diff2 = (elev_i - elev_j)**2
ddem_diff2 = (ddem_i - ddem_j)**2 # nans present - handle below

# elevation differences
elev_diffs = np.sqrt(elev_diff2)
print('elevation neighbor differences (meters):')
for p in [50, 75, 90, 95, 99]:
    print(f' p{p:02d}: {np.percentile(elev_diffs, p):.4f} m')

print()

# ddem differences (excluding nan edges)
valid_ddem = ~(np.isnan(ddem_i) | np.isnan(ddem_j))
ddem_diffs = np.sqrt(ddem_diff2[valid_ddem])
print('dDEM neighbour differences (metres):')
for p in [50, 75, 90, 95, 99]:
    print(f'  p{p:02d}: {np.percentile(ddem_diffs, p):.4f} m')

elevation neighbor differences (meters):
 p50: 0.1604 m
 p75: 0.3306 m
 p90: 0.5346 m
 p95: 0.6716 m
 p99: 0.9683 m

dDEM neighbour differences (metres):
  p50: 0.1730 m
  p75: 0.3416 m
  p90: 0.5671 m
  p95: 0.7426 m
  p99: 1.1923 m


In [24]:
# for given beta, what fraction of edges have weight <0.5?
# w < 0.5 means exp(-beta * d^2) < 0.5
# i.e. beta * d^2 > ln(2) ~ 0.693
def frac_below_half(diffs_squared, beta):
    w = np.exp(-beta * diffs_squared)
    return 100 * (w < 0.5).mean()

print('Fraction of edges with weight < 0.5 at candidate beta values:')
print(f'{"beta":>10}  {"elev (%)":>10}  {"ddem (%)":>10}')
print('-' * 34)
for beta in [0.005, 0.05, 0.5, 2.0, 5.0, 10.0]:
    fe = frac_below_half(elev_diff2, beta)
    fd = frac_below_half(ddem_diff2[valid_ddem], beta)
    print(f'{beta:>10.3f}  {fe:>10.2f}  {fd:>10.2f}')

Fraction of edges with weight < 0.5 at candidate beta values:
      beta    elev (%)    ddem (%)
----------------------------------
     0.005        0.00        0.00
     0.050        0.01        0.01
     0.500        0.33        1.05
     2.000        7.65        9.17
     5.000       20.94       22.06
    10.000       33.04       34.48


In [25]:
### update beta values and recompute weights ###
###                                          ###

In [26]:
# Updated beta values informed by empirical difference distributions
BETA_ELEV   = 10.0   # sigma ~ 0.22m, suppresses ~33% of edges
BETA_ASPECT =  2.0   # unchanged -- already well calibrated
BETA_DDEM   = 10.0   # same as elev -- similar difference distribution

w_elev = np.exp(-BETA_ELEV * elev_diff2).astype(np.float32)

w_aspect = np.exp(-BETA_ASPECT * aspect_dist2).astype(np.float32)

w_ddem = np.exp(-BETA_DDEM * ddem_diff2).astype(np.float32)
w_ddem[ddem_nan_mask] = 0.0

print('Updated edge weight statistics:')
print(f'{"":12}  {"Min":>8}  {"Max":>8}  {"Mean":>8}  {"<0.5 edges":>12}')
print('-' * 56)
for name, w in [('w_elev', w_elev), ('w_aspect', w_aspect), ('w_ddem', w_ddem)]:
    below_half = (w < 0.5).sum()
    print(f'{name:12}  {w.min():>8.4f}  {w.max():>8.4f}  {w.mean():>8.4f}  {below_half:>12,}')

Updated edge weight statistics:
                   Min       Max      Mean    <0.5 edges
--------------------------------------------------------
w_elev          0.0000    1.0000    0.6414     5,386,934
w_aspect        0.0003    1.0000    0.7666     3,297,752
w_ddem          0.0000    1.0000    0.6195     5,672,700


In [28]:
### build sparse adjacency matrices and normalized laplacians ###
###                                                           ###

In [43]:
N = arr.shape[0]

def build_normalized_laplacian(edge_i, edge_j, weights, n):
    """
    build normalized symmetric graph laplacian from edge lists
    L = I - D^{-1/2} A D^{-1/2} # A the adjacency matrix
    eigenvalues lie in [0, 2].
    """
    # build symm. adj. matrix from edge list
    # each undirected edge already appears twice (directions)
    A = coo_matrix(
        (weights, (edge_i, edge_j)),
        shape=(n,n)
    ).tocsr()

    # degree vector: row sums of adjacency matrix
    d = np.array(A.sum(axis=1)).ravel()

    # avoid divide by zero by clamping d before sqrt
    d_safe = np.where(d > 0, d, 1.0) # replace zeros with 1.0 temporarily

    # D^{-1/2}: avoid division by zero for isolated nodes
    d_invsqrt = np.where(d > 0, 1.0 / np.sqrt(d_safe), 0.0)

    # diagonal matrix D^{-1/2} as sparse
    D_invsqrt = csr_matrix(
        (d_invsqrt, (np.arange(n), np.arange(n))),
        shape=(n,n)
    )

    # normalized laplacian
    L = speye(n, format='csr') - D_invsqrt @ A @ D_invsqrt

    # Report how many nodes were treated as isolated
    n_isolated = (d == 0).sum()
    print(f'  Isolated nodes (d=0): {n_isolated}')

    return L, d

    return L, d

In [41]:
print("building L_elev...")
L_elev, d_elev = build_normalized_laplacian(edge_i, edge_j, w_elev, N)
print(f' nnz: {L_elev.nnz:,}')

building L_elev...
  Isolated nodes (d=0): 1
 nnz: 18,345,196


In [44]:
print("building L_aspect...")
L_aspect, d_aspect = build_normalized_laplacian(edge_i, edge_j, w_aspect, N)
print(f' nnz: {L_aspect.nnz:,}')

building L_aspect...
  Isolated nodes (d=0): 0
 nnz: 18,347,260


In [45]:
print("building L_ddem...")
L_ddem, d_ddem = build_normalized_laplacian(edge_i, edge_j, w_ddem, N)
print(f' nnz: {L_ddem.nnz:,}')

building L_ddem...
  Isolated nodes (d=0): 9065
 nnz: 18,267,646


In [46]:
import scipy.sparse as sp

# Laplacians -- most expensive to rebuild
sp.save_npz('L_elev.npz',   L_elev)
sp.save_npz('L_aspect.npz', L_aspect)
sp.save_npz('L_ddem.npz',   L_ddem)

# Edge list -- needed to rebuild Laplacians with different beta
np.save('edge_i.npy', edge_i)
np.save('edge_j.npy', edge_j)

# dDEM node array -- required careful alignment, worth preserving
np.save('ddem_nodes.npy', ddem_nodes)

# Beta values -- document the calibration decisions
beta_params = {
    'BETA_ELEV':   10.0,
    'BETA_ASPECT':  2.0,
    'BETA_DDEM':   10.0,
    'note': 'calibrated from empirical neighbour difference distributions'
}
np.save('beta_params.npy', beta_params, allow_pickle=True)

print('Saved:')
print('  L_elev.npz, L_aspect.npz, L_ddem.npz')
print('  edge_i.npy, edge_j.npy')
print('  ddem_nodes.npy')
print('  beta_params.npy')

Saved:
  L_elev.npz, L_aspect.npz, L_ddem.npz
  edge_i.npy, edge_j.npy
  ddem_nodes.npy
  beta_params.npy
